<a href="https://colab.research.google.com/github/tvishaaprabhu/reef-check/blob/main/reef_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪸 Reef Check — no pretrained models

Runtime → Change runtime type → **T4 GPU** (optional, a CPU also works, just slower), then run the cells in order. Open the public `gradio.live` link printed at the end.

**Everything in this notebook is trained or computed from scratch — nothing downloads pretrained weights:**

| Task | Approach (no pretrained model) |
|---|---|
| Colony **detection** | SLIC superpixels + region-adjacency-graph color merging, foreground/background split with a `KMeans` fit fresh on every single image |
| Colony **health** | Rule-based scoring from HSV color statistics inside each colony mask (bleached-white %, algae/dead %, vivid living-tissue %) |
| Colony **type** (growth form) | Shape/texture descriptors (compactness, solidity, skeleton branching density, GLCM texture) fed into a hand-written decision rule |
| **Super-resolution (4×)** | A small CNN (`TinySRNet`) with **randomly-initialized weights**, trained for a few hundred steps *on the uploaded image/video itself* (self-supervised "zero-shot" internal learning) before being used to upscale it |
| Cross-frame **tracking** | IoU-based greedy tracker (classical, no learning) |

⚠️ Because there is no pretrained detector/classifier behind this, accuracy will be lower and more lighting/framing-sensitive than a model like FastSAM + CLIP would give. Sliders are exposed so you can tune it to your footage. Colony "type" here means a coarse **growth-form bucket**, labeled with a typical genus for that form (branching/table: Acropora, massive: Porites, plate/encrusting: Montipora, cup/scroll: Turbinaria, soft: Sinularia, fire coral: Millepora) — true species-level ID would need a labeled dataset and a trained (not pretrained) classifier, which you can bolt on later using the same feature vectors this notebook already extracts (`shape_descriptors`, texture, color) if you collect labels.


In [ ]:
!pip -q install -U scikit-image scikit-learn "imageio[ffmpeg]" gradio

In [ ]:
import os, cv2, math, uuid, traceback, tempfile, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, gradio as gr, imageio
from PIL import Image
from sklearn.cluster import KMeans
from skimage.segmentation import slic
try:
    from skimage import graph as skgraph          # scikit-image >= 0.20
except ImportError:
    from skimage.future import graph as skgraph    # older scikit-image
from skimage.measure import regionprops, label as sklabel
from skimage.morphology import skeletonize, remove_small_objects, closing, disk
from skimage.feature import graycomatrix, graycoprops

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT = tempfile.mkdtemp()
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Growth-form categories a coral colony is bucketed into, purely from shape/texture
# descriptors. Each category is labeled with a typical genus for that growth form;
# this is a growth-form guess, not species-level identification (no pretrained
# classifier is used anywhere in this notebook).
TYPES = [
    ("Acropora",   "Branching / table coral"),
    ("Porites",    "Massive / boulder coral"),
    ("Montipora",  "Plate / encrusting coral"),
    ("Turbinaria", "Cup / scroll coral"),
    ("Sinularia",  "Soft / leather coral"),
    ("Millepora",  "Fire coral"),
]

COL = {
    "Healthy": (46, 204, 113),
    "Partially bleached / stressed": (241, 196, 15),
    "Bleached": (236, 240, 241),
    "Dead / algae-covered": (231, 76, 60),
}


In [ ]:
def fit(rgb, max_side=1280):
    h, w = rgb.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1:
        rgb = cv2.resize(rgb, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    h, w = rgb.shape[:2]
    return np.ascontiguousarray(rgb[: h - h % 2, : w - w % 2])


def enhance(rgb):
    """Grey-world white balance + CLAHE contrast boost. Classical CV, no learned weights."""
    f = rgb.astype(np.float32)
    m = f.reshape(-1, 3).mean(0)
    g = np.clip(m.mean() / (m + 1e-6), 0.5, 3.0)
    f = np.clip(f * g, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(f, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(2.0, (8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def _weight_mean_color(graph_, src, dst, n):
    diff = graph_.nodes[dst]["mean color"] - graph_.nodes[n]["mean color"]
    return {"weight": np.linalg.norm(diff)}


def _merge_mean_color(graph_, src, dst):
    graph_.nodes[dst]["total color"] += graph_.nodes[src]["total color"]
    graph_.nodes[dst]["pixel count"] += graph_.nodes[src]["pixel count"]
    graph_.nodes[dst]["mean color"] = graph_.nodes[dst]["total color"] / graph_.nodes[dst]["pixel count"]


def superpixel_regions(rgb, n_segments=260, compactness=14, merge_thr=26):
    """SLIC superpixels merged by color similarity via a region-adjacency graph.
    Purely classical / unsupervised - fit fresh on every image, no downloaded weights."""
    segs = slic(rgb, n_segments=n_segments, compactness=compactness, sigma=1, start_label=1, channel_axis=-1)
    g = skgraph.rag_mean_color(rgb, segs)
    merged = skgraph.merge_hierarchical(segs, g, thresh=merge_thr, rag_copy=False,
                                         in_place_merge=True, merge_func=_merge_mean_color,
                                         weight_func=_weight_mean_color)
    return merged


def _edge_density(gray, mask):
    e = cv2.Canny(gray, 40, 120)
    return float(e[mask].mean() / 255.0) if mask.any() else 0.0


def segment_colonies(rgb, min_area_frac=0.004, max_area_frac=0.45, texture_gate=0.02):
    """Turn merged superpixel regions into candidate coral-colony masks.

    Foreground/background split is done with a KMeans(k=3) fit on each region's own
    mean Lab color + edge-texture - i.e. it is trained fresh on this single image,
    never on an external dataset, so there is no pretrained model involved.
    """
    h, w = rgb.shape[:2]
    merged = superpixel_regions(rgb)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    ids = np.unique(merged)
    feats, masks, areas = [], [], []
    for rid in ids:
        m = merged == rid
        area = m.mean()
        if area < min_area_frac or area > max_area_frac:
            continue
        mlab = lab[m].mean(0)
        mhsv = hsv[m].mean(0)
        tex = _edge_density(gray, m)
        feats.append([mlab[0] / 255, mlab[1] / 255, mlab[2] / 255, mhsv[0] / 180, mhsv[1] / 255, tex])
        masks.append(m)
        areas.append(area)
    if len(masks) < 2:
        return []

    X = np.array(feats)
    k = min(3, len(masks))
    km = KMeans(n_clusters=k, n_init=6, random_state=0).fit(X)
    lbl = km.labels_

    # The background cluster is whichever cluster covers most of the frame area
    # AND is texturally smooth (low edge density) - typical of open water / sand.
    clust_area = np.array([sum(a for a, l in zip(areas, lbl) if l == c) for c in range(k)])
    clust_tex = np.array([np.mean([f[5] for f, l in zip(feats, lbl) if l == c]) for c in range(k)])
    bg_cluster = int(np.argmax(clust_area - 0.5 * clust_tex))

    out = []
    for m, a, l, f in zip(masks, areas, lbl, feats):
        is_water_hue = 0.5 < f[3] < 0.78 and f[4] < 0.55  # blue-cyan hue, muted saturation
        if l == bg_cluster or (is_water_hue and f[5] < texture_gate):
            continue
        out.append(m)
    out = sorted(out, key=lambda m: -m.mean())
    return out[:60]


def box_of(m, pad=0.08):
    ys, xs = np.where(m)
    h, w = m.shape
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    px, py = int((x1 - x0) * pad), int((y1 - y0) * pad)
    return int(max(x0 - px, 0)), int(max(y0 - py, 0)), int(min(x1 + px, w - 1)), int(min(y1 + py, h - 1))


def box_iou(a, b):
    ix, iy = max(0, min(a[2], b[2]) - max(a[0], b[0])), max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0


def contained(a, b, sa, sb):
    return np.logical_and(a, b).sum() / max(1, min(sa, sb))


In [ ]:
def color_fractions(hsv_px):
    """Fraction of a colony's pixels that look bleached-white, algae/dead, or vividly alive."""
    h, s, v = hsv_px[:, 0], hsv_px[:, 1], hsv_px[:, 2]
    white = (s < 45) & (v > 150)
    algae = (((h > 30) & (h < 95)) | (s < 60) & (v < 110)) & ~white  # green algae or dark grey rubble
    vivid = (s > 70) & (v > 40) & (v < 235) & ~white & ~algae
    return float(white.mean()), float(algae.mean()), float(vivid.mean())


def health_score(hsv_px):
    white_f, algae_f, vivid_f = color_fractions(hsv_px)
    score = 100 * (0.6 * vivid_f + 0.25 * (1 - white_f) + 0.15 * (1 - algae_f))
    return float(np.clip(score, 0, 100)), white_f, algae_f, vivid_f


def status_of(score, algae_f, white_f):
    if algae_f > 0.35 and score < 55:
        return "Dead / algae-covered"
    if score >= 70:
        return "Healthy"
    if score >= 45:
        return "Partially bleached / stressed"
    return "Bleached"


def shape_descriptors(mask):
    """Morphology-only descriptors used to bucket a colony into a growth form."""
    m = mask.astype(np.uint8)
    m = closing(m, disk(2)).astype(np.uint8)
    props = regionprops(m)
    if not props:
        return dict(solidity=1, eccentricity=0, compactness=1, branch_density=0, extent=1)
    p = props[0]
    perim = max(p.perimeter, 1e-3)
    compactness = float(np.clip(4 * math.pi * p.area / (perim ** 2), 0, 1))
    skel = skeletonize(m.astype(bool))
    # 3x3 neighbour-count convolution to find skeleton branch/end points
    if skel.sum() > 0:
        nb = cv2.filter2D(skel.astype(np.uint8), -1, np.ones((3, 3), np.uint8)) * skel
        branch_pts = int(((nb >= 4)).sum())
        branch_density = branch_pts / max(skel.sum(), 1)
    else:
        branch_density = 0.0
    return dict(solidity=float(p.solidity), eccentricity=float(p.eccentricity),
                compactness=compactness, branch_density=float(branch_density), extent=float(p.extent))


def texture_softness(gray, mask):
    """GLCM homogeneity/contrast inside the mask - crude proxy for a fleshy/soft-looking
    colony surface (soft corals) vs a hard, sharply-edged skeleton (hard corals)."""
    ys, xs = np.where(mask)
    if len(ys) < 8:
        return 0.5, 0.5
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    patch = gray[y0:y1 + 1, x0:x1 + 1]
    patch = cv2.resize(patch, (64, 64), interpolation=cv2.INTER_AREA)
    glcm = graycomatrix(patch, distances=[2], angles=[0], levels=256, symmetric=True, normed=True)
    return float(graycoprops(glcm, "homogeneity")[0, 0]), float(graycoprops(glcm, "contrast")[0, 0])


def classify_type(mask, gray):
    d = shape_descriptors(mask)
    homog, contrast = texture_softness(gray, mask)
    scores = np.zeros(len(TYPES))
    # Branching: low compactness, high skeleton branch density, elongated
    scores[0] = (1 - d["compactness"]) * 0.6 + min(d["branch_density"] * 4, 1) * 0.4
    # Massive/boulder: round, solid, compact
    scores[1] = d["compactness"] * 0.6 + d["solidity"] * 0.4
    # Encrusting/plate: high extent (fills bbox), low eccentricity, flattish
    scores[2] = d["extent"] * 0.5 + (1 - d["eccentricity"]) * 0.3 + (1 - min(d["branch_density"] * 4, 1)) * 0.2
    # Foliose: medium solidity, irregular edges, moderate compactness
    scores[3] = (1 - abs(d["solidity"] - 0.7)) * 0.5 + (1 - d["compactness"]) * 0.3 + homog * 0.2
    # Soft/leathery: smooth homogeneous texture (low contrast), fuzzy irregular boundary
    scores[4] = homog * 0.55 + (1 - d["solidity"]) * 0.25 + (1 - d["compactness"]) * 0.2
    # Mushroom/free-living: high compactness AND high eccentricity-independent roundness, small
    scores[5] = d["compactness"] * 0.5 + d["solidity"] * 0.3 + (1 - min(d["branch_density"] * 4, 1)) * 0.2
    k = int(np.argmax(scores))
    conf = float(scores[k] / (scores.sum() + 1e-6))
    return k, np.clip(conf, 0.2, 0.95), d


In [ ]:
def analyze_frame(rgb, min_area, max_col):
    masks = segment_colonies(rgb, min_area_frac=min_area)
    if not masks:
        return []
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    boxes = [box_of(m) for m in masks]
    sums = [m.sum() for m in masks]

    # drop masks that sit almost entirely inside a larger already-kept one
    kept = []
    for i in range(len(masks)):
        if len(kept) >= max_col:
            break
        if all(box_iou(boxes[i], boxes[j]) == 0 or contained(masks[i], masks[j], sums[i], sums[j]) < 0.6 for j in kept):
            kept.append(i)

    out = []
    for i in kept:
        m, (x0, y0, x1, y1) = masks[i], boxes[i]
        px = hsv[m]
        score, white_f, algae_f, vivid_f = health_score(px)
        tk, tconf, _ = classify_type(m, gray)
        out.append(dict(mask=m, box=boxes[i], raw=rgb[y0:y1 + 1, x0:x1 + 1].copy(),
                         health=score, white=white_f, algae=algae_f, vivid=vivid_f,
                         type_k=tk, type_conf=tconf, area=float(m.mean())))
    return out


class Tracker:
    def __init__(s):
        s.last, s.tracks, s.nid, s.step = {}, {}, 1, 0

    def update(s, dets, t):
        s.step += 1
        used = set()
        for d in sorted(dets, key=lambda d: -d["area"]):
            best, bid = 0.25, None
            for tid, (b, st) in s.last.items():
                if tid in used or s.step - st > 3:
                    continue
                v = box_iou(d["box"], b)
                if v > best:
                    best, bid = v, tid
            if bid is None:
                bid = s.nid
                s.nid += 1
            used.add(bid)
            d["id"] = bid
            s.last[bid] = (d["box"], s.step)
            tt = s.tracks.setdefault(bid, {"tp": np.zeros(len(TYPES)), "h": [], "w": [], "a": [], "n": 0,
                                            "best": None, "barea": 0.0, "first": t})
            tt["tp"][d["type_k"]] += d["type_conf"]
            tt["h"].append(d["health"])
            tt["w"].append(d["white"])
            tt["a"].append(d["algae"])
            tt["n"] += 1
            if d["area"] > tt["barea"]:
                tt["barea"], tt["best"] = d["area"], d["raw"]


def draw(rgb, dets):
    img = rgb.copy()
    ov = img.copy()
    for d in dets:
        ov[d["mask"]] = COL[d["status"]]
    img = cv2.addWeighted(ov, 0.35, img, 0.65, 0)
    sc = max(0.45, img.shape[1] / 1800)
    for d in dets:
        c = COL[d["status"]]
        cs, _ = cv2.findContours(d["mask"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img, cs, -1, c, 2)
        x0, y0 = d["box"][:2]
        t = f'#{d["id"]} {d["label"]} {d["health"]:.0f}%'
        (tw, th), _ = cv2.getTextSize(t, cv2.FONT_HERSHEY_SIMPLEX, sc, 1)
        yt = max(y0, th + 8)
        cv2.rectangle(img, (x0, yt - th - 8), (x0 + tw + 6, yt), c, -1)
        cv2.putText(img, t, (x0 + 3, yt - 5), cv2.FONT_HERSHEY_SIMPLEX, sc, (0, 0, 0), 1, cv2.LINE_AA)
    return img


In [ ]:
class TinySRNet(nn.Module):
    """A small residual pixel-shuffle CNN. Weights are randomly initialized and trained
    from scratch on the uploaded image/video itself - no checkpoint is ever loaded."""

    def __init__(self, scale=4, ch=48):
        super().__init__()
        self.scale = scale
        self.body = nn.Sequential(
            nn.Conv2d(3, ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(ch, 3 * scale * scale, 3, padding=1),
        )
        self.up = nn.PixelShuffle(scale)

    def forward(self, x):
        base = F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)
        r = self.up(self.body(x))
        return torch.clamp(base + r, 0, 1)


def _sample_patch_pair(frame, scale, hr_size):
    h, w = frame.shape[:2]
    if h < hr_size or w < hr_size:
        frame = cv2.resize(frame, (max(w, hr_size), max(h, hr_size)), interpolation=cv2.INTER_CUBIC)
        h, w = frame.shape[:2]
    y = np.random.randint(0, h - hr_size + 1)
    x = np.random.randint(0, w - hr_size + 1)
    hr = frame[y:y + hr_size, x:x + hr_size]
    lr = cv2.GaussianBlur(hr, (3, 3), 0.6)
    lr = cv2.resize(lr, (hr_size // scale, hr_size // scale), interpolation=cv2.INTER_AREA)
    return lr, hr


def train_sr_net(frames, scale=4, steps=250, patch=96, batch=8, lr=2e-4, progress=None):
    """Self-supervised 'zero-shot' super-resolution: the network only ever sees
    down-sampled/blurred crops of the user's OWN frames as input, and the original
    crops as the target - so it learns this video's texture statistics from scratch."""
    net = TinySRNet(scale=scale).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    torch.set_grad_enabled(True)
    net.train()
    for step in range(steps):
        lrs, hrs = [], []
        for _ in range(batch):
            f = frames[np.random.randint(len(frames))]
            lr_p, hr_p = _sample_patch_pair(f, scale, patch)
            lrs.append(lr_p.astype(np.float32) / 255.0)
            hrs.append(hr_p.astype(np.float32) / 255.0)
        xb = torch.from_numpy(np.stack(lrs)).permute(0, 3, 1, 2).to(device)
        yb = torch.from_numpy(np.stack(hrs)).permute(0, 3, 1, 2).to(device)
        pred = net(xb)
        loss = F.l1_loss(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if progress is not None and step % 10 == 0:
            progress(step / steps, desc=f"Training super-resolution net (step {step}/{steps}, loss {loss.item():.4f})")
    net.eval()
    torch.set_grad_enabled(False)
    return net


@torch.no_grad()
def sr_tile(net, img, scale):
    x = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
    y = net(x).squeeze(0).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    return (y * 255).round().astype(np.uint8)


@torch.no_grad()
def upscale(net, img, scale=4, max_side=512, tile=160, pad=8):
    h, w = img.shape[:2]
    s = min(1.0, max_side / max(h, w))
    if s < 1:
        img = cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]
    out = np.zeros((h * scale, w * scale, 3), np.uint8)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            y0, x0 = max(y - pad, 0), max(x - pad, 0)
            y1, x1 = min(y + tile + pad, h), min(x + tile + pad, w)
            t = sr_tile(net, np.ascontiguousarray(img[y0:y1, x0:x1]), scale)
            hh, ww = (min(y + tile, h) - y) * scale, (min(x + tile, w) - x) * scale
            oy, ox = (y - y0) * scale, (x - x0) * scale
            out[y * scale:y * scale + hh, x * scale:x * scale + ww] = t[oy:oy + hh, ox:ox + ww]
    return out


In [ ]:
def video_iter(path, afps, max_frames):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, int(round(fps / afps)))
    n = min(max_frames, math.ceil(total / step)) if total > 0 else max_frames

    def gen():
        i = k = 0
        while k < n:
            if not cap.grab():
                break
            if i % step == 0:
                ok, f = cap.retrieve()
                if not ok:
                    break
                yield i, i / fps, fit(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
                k += 1
            i += 1
        cap.release()

    return gen(), n


def run(file, afps, max_frames, min_area, max_col, do_sr, sr_steps, progress=gr.Progress()):
    try:
        return analyze(file, afps, max_frames, min_area, max_col, do_sr, sr_steps, progress)
    except gr.Error:
        raise
    except Exception as ex:
        traceback.print_exc()
        raise gr.Error(f"{type(ex).__name__}: {ex}")


DET_COLS = ["Frame", "Time (s)", "Colony ID", "Type (this frame)", "Type confidence %", "Health %", "Status",
            "Vivid tissue %", "White/bleached %", "Algae/dead %", "Colony area % of frame", "Box (x0,y0,x1,y1)"]
COL_COLS = ["Colony ID", "Colony type", "Type confidence %", "Frames seen", "First seen (s)",
            "Mean health %", "Min health %", "Max health %", "Status"]


def analyze(file, afps, max_frames, min_area_pct, max_col, do_sr, sr_steps, progress):
    if file is None:
        raise gr.Error("Please upload a coral reef image or video.")
    path = file if isinstance(file, str) else file.name
    uid = uuid.uuid4().hex[:8]
    is_img = os.path.splitext(path)[1].lower() in IMG_EXT
    if is_img:
        raw_frames = [(0, 0.0, fit(np.array(Image.open(path).convert("RGB"))))]
        total = 1
    else:
        gen, total = video_iter(path, float(afps), int(max_frames))
        raw_frames = list(gen)

    if not raw_frames:
        raise gr.Error("Could not read any frames from this file.")

    enhanced = [(fi, t, enhance(rgb)) for fi, t, rgb in raw_frames]

    sr_net = None
    if do_sr:
        sample = [e for _, _, e in enhanced[:: max(1, len(enhanced) // 6)]][:6]
        progress(0.0, desc="Training self-supervised super-resolution net on this input...")
        sr_net = train_sr_net(sample, scale=4, steps=int(sr_steps), progress=progress)

    tr = Tracker()
    det_rows, frame_rows = [], []
    best_ann, best_n, writer, vpath = None, -1, None, None
    keys = []
    kstep = max(1, total // 4)
    min_area = float(min_area_pct) / 100.0

    for n, (fi, t, e) in enumerate(enhanced):
        progress(n / max(total, 1), desc=f"Analyzing frame {n + 1}/{total}")
        dets = analyze_frame(e, min_area, int(max_col))
        tr.update(dets, t)
        cover = np.zeros(e.shape[:2], bool)
        for d in dets:
            tt = tr.tracks[d["id"]]
            best_k = int(np.argmax(tt["tp"]))
            d["label"] = TYPES[best_k][0]
            d["status"] = status_of(d["health"], d["algae"], d["white"])
            cover |= d["mask"]
            det_rows.append({
                "Frame": fi, "Time (s)": round(t, 2), "Colony ID": d["id"],
                "Type (this frame)": f"{TYPES[d['type_k']][1]} ({TYPES[d['type_k']][0]})",
                "Type confidence %": round(100 * d["type_conf"], 1), "Health %": round(d["health"], 1),
                "Status": d["status"], "Vivid tissue %": round(100 * d["vivid"], 1),
                "White/bleached %": round(100 * d["white"], 1), "Algae/dead %": round(100 * d["algae"], 1),
                "Colony area % of frame": round(100 * d["area"], 2), "Box (x0,y0,x1,y1)": str(d["box"]),
            })
        ann = draw(e, dets)
        if not is_img:
            if writer is None:
                vpath = os.path.join(OUT, f"reef_check_{uid}.mp4")
                writer = imageio.get_writer(vpath, fps=float(afps), codec="libx264", quality=8, macro_block_size=1)
            writer.append_data(ann)
        frame_rows.append({
            "Frame": fi, "Time (s)": round(t, 2), "Colonies": len(dets),
            "Mean health %": round(float(np.mean([d["health"] for d in dets])), 1) if dets else np.nan,
            "Coral cover %": round(100 * float(cover.mean()), 1),
            "Healthy": sum(d["status"] == "Healthy" for d in dets),
            "Stressed": sum(d["status"] == "Partially bleached / stressed" for d in dets),
            "Bleached": sum(d["status"] == "Bleached" for d in dets),
            "Dead": sum(d["status"] == "Dead / algae-covered" for d in dets),
        })
        if len(dets) > best_n:
            best_n, best_ann = len(dets), ann
        if n % kstep == 0 and len(keys) < 4:
            keys.append((t, e))
    if writer:
        writer.close()

    col_rows = []
    for tid, tt in tr.tracks.items():
        k = int(np.argmax(tt["tp"]))
        mh = float(np.mean(tt["h"]))
        mw, ma = float(np.mean(tt["w"])), float(np.mean(tt["a"]))
        tt["type"] = f"{TYPES[k][1]} ({TYPES[k][0]})"
        tt["mh"] = mh
        tt["status"] = status_of(mh, ma, mw)
        col_rows.append({
            "Colony ID": tid, "Colony type": tt["type"],
            "Type confidence %": round(100 * tt["tp"][k] / tt["n"], 1), "Frames seen": tt["n"],
            "First seen (s)": round(tt["first"], 2), "Mean health %": round(mh, 1),
            "Min health %": round(float(np.min(tt["h"])), 1), "Max health %": round(float(np.max(tt["h"])), 1),
            "Status": tt["status"],
        })

    ddf, fdf = pd.DataFrame(det_rows, columns=DET_COLS), pd.DataFrame(frame_rows)
    cdf = pd.DataFrame(col_rows, columns=COL_COLS).sort_values("Colony ID")

    fgal, cgal = [], []
    for j, (t, e) in enumerate(keys):
        progress(j / max(len(keys), 1), desc="Super-resolving key frames")
        p = os.path.join(OUT, f"{uid}_frame_{j}.png")
        Image.fromarray(upscale(sr_net, e, 4, 512) if do_sr else e).save(p)
        fgal.append((p, f"t = {t:.1f}s · {'4x self-trained super-resolution' if do_sr else 'enhanced'}"))

    top = sorted(tr.tracks.items(), key=lambda x: -x[1]["n"] * x[1]["barea"])[:12]
    for j, (tid, tt) in enumerate(top):
        progress(j / max(len(top), 1), desc="Super-resolving colonies")
        p = os.path.join(OUT, f"{uid}_colony_{tid}.png")
        Image.fromarray(upscale(sr_net, tt["best"], 4, 256) if do_sr else tt["best"]).save(p)
        cgal.append((p, f"#{tid} · {tt['type']} · {tt['mh']:.0f}% · {tt['status']}"))

    csv = os.path.join(OUT, f"reef_check_{uid}.csv")
    ddf.to_csv(csv, index=False)

    sc = cdf["Status"].value_counts() if len(cdf) else pd.Series(dtype=int)
    types = ", ".join(f"{a} ({b})" for a, b in cdf["Colony type"].value_counts().head(3).items()) if len(cdf) else "-"
    overall = cdf["Mean health %"].mean() if len(cdf) else float("nan")
    grade = "-" if np.isnan(overall) else "Good" if overall >= 70 else "Fair" if overall >= 50 else "Poor" if overall >= 30 else "Critical"
    summ = (f"### Reef summary\n| Metric | Value |\n|---|---|\n| Frames analyzed | {len(fdf)} |\n"
            f"| Coral colonies found | {len(cdf)} |\n| Overall reef health | **{overall:.1f}%** ({grade}) |\n"
            f"| Average coral cover | {fdf['Coral cover %'].mean():.1f}% |\n"
            f"| Healthy / Stressed / Bleached / Dead | {sc.get('Healthy', 0)} / {sc.get('Partially bleached / stressed', 0)} / "
            f"{sc.get('Bleached', 0)} / {sc.get('Dead / algae-covered', 0)} |\n"
            f"| Most common colony types | {types} |\n")
    if not len(cdf):
        summ += ("\n⚠️ **No coral colonies were detected.** This pipeline is fully classical/self-trained "
                 "(no pretrained weights), so it is more sensitive to lighting and framing than a pretrained "
                 "detector would be - try lowering *Minimum colony size* or check the footage actually shows coral close-up.\n")

    plot_df = fdf.dropna(subset=["Mean health %"])
    if not len(plot_df):
        plot_df = pd.DataFrame({"Time (s)": [0.0], "Mean health %": [0.0]})

    return best_ann, vpath, summ, plot_df, fdf, ddf, cdf, cgal, fgal, csv


In [ ]:
CSS = """
.gradio-container{max-width:1300px!important;margin:auto}
#hero{background:linear-gradient(135deg,#03045e,#0077b6 50%,#00b4d8);padding:30px 34px;border-radius:20px;margin-bottom:8px}
#hero h1{color:#fff;margin:0;font-size:2.3rem;letter-spacing:.5px}
#hero p{color:#caf0f8;margin:8px 0 0;font-size:1.05rem}
.legend span{display:inline-block;width:12px;height:12px;border-radius:3px;margin:0 6px 0 12px;vertical-align:middle;border:1px solid #999}
"""
LEGEND = ('<div class="legend"><b>Legend:</b><span style="background:#2ecc71"></span>Healthy'
          '<span style="background:#f1c40f"></span>Partially bleached / stressed'
          '<span style="background:#ecf0f1"></span>Bleached<span style="background:#e74c3c"></span>Dead / algae-covered</div>')

with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan", secondary_hue="blue"), css=CSS, title="Reef Check") as demo:
    gr.HTML('<div id="hero"><h1>🪸 Reef Check</h1>'
            '<p>Upload a coral reef image or video to detect colonies, assess health frame by frame, '
            'identify colony type and get 4× super-resolved outputs.</p></div>')
    with gr.Row():
        with gr.Column(scale=1):
            inp = gr.File(label="Coral reef image or video", file_types=["image", "video"])
            with gr.Accordion("Settings", open=False):
                afps = gr.Slider(0.5, 5, 1, step=0.5, label="Frames analyzed per second (video)")
                maxf = gr.Slider(5, 200, 40, step=5, label="Max frames to analyze")
                min_area = gr.Slider(0.2, 5.0, 0.6, step=0.1, label="Minimum colony size (% of frame area)")
                maxc = gr.Slider(1, 40, 20, step=1, label="Max colonies per frame")
                sr = gr.Checkbox(True, label="Super-resolve outputs (4x, self-trained per upload)")
                sr_steps = gr.Slider(50, 600, 250, step=50, label="Super-resolution training steps")
            btn = gr.Button("Analyze reef", variant="primary", size="lg")
            gr.HTML(LEGEND)
            csv = gr.File(label="Download full per-frame results (CSV)")
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("Detections"):
                    out_img = gr.Image(label="Annotated frame (most colonies)", type="numpy")
                    out_vid = gr.Video(label="Annotated video")
                with gr.Tab("Report"):
                    summ = gr.Markdown()
                    plot = gr.LinePlot(x="Time (s)", y="Mean health %", title="Reef health over time", height=280)
                    ctab = gr.Dataframe(label="Colonies (tracked across frames)", wrap=True)
                with gr.Tab("Frame-wise"):
                    ftab = gr.Dataframe(label="Per-frame summary", wrap=True)
                    dtab = gr.Dataframe(label="Per-frame colony health and type", wrap=True)
                with gr.Tab("Super-resolution"):
                    cgal = gr.Gallery(label="Colonies (4x upscaled)", columns=3, height="auto", object_fit="contain")
                    fgal = gr.Gallery(label="Key frames (4x upscaled)", columns=2, height="auto", object_fit="contain")
    btn.click(run, [inp, afps, maxf, min_area, maxc, sr, sr_steps],
              [out_img, out_vid, summ, plot, ftab, dtab, ctab, cgal, fgal, csv])

demo.queue().launch(share=True, debug=True, show_error=True)
